# FLUX txt2img — fp8 and fp16 on Kaggle P100
**Run on Kaggle with GPU (P100 — 16GB VRAM, 29GB RAM)**
Run cells in order. Do not skip any cell.

In [ ]:
# CELL 1: Install — RESTART RUNTIME after this
!pip install -q \
    diffusers==0.36.0 \
    transformers==4.49.0 \
    accelerate \
    sentencepiece \
    protobuf \
    Pillow
!pip install -q "bitsandbytes>=0.46.1"
print('RESTART RUNTIME')

In [1]:
import bitsandbytes
print(bitsandbytes.__version__) 

0.50.0


In [2]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))

# Test bitsandbytes separately
try:
    import bitsandbytes as bnb
    print('bitsandbytes:', bnb.__version__)
    print('bitsandbytes OK')
except Exception as e:
    print('bitsandbytes FAILED:', e)

PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4
bitsandbytes: 0.50.0
bitsandbytes OK


In [3]:
# CELL 2: Setup
import os

os.environ['HF_TOKEN']                  = 'your_token_here'
os.environ['HUGGING_FACE_HUB_TOKEN']    = 'your_token_here'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF']   = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM']    = 'false'
os.environ['FLUX_OFFLOAD_DIR']          = '/kaggle/working/offload_flux'
os.makedirs('/kaggle/working/offload_flux', exist_ok=True)

import torch, shutil
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
t,u,f = shutil.disk_usage('/kaggle/working')
print(f'Disk: {f/1e9:.0f}GB free of {t/1e9:.0f}GB')

CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB
Disk: 21GB free of 21GB


In [4]:
# CELL 3: HuggingFace login
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
print('Logged in')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in


In [5]:
# CELL 4: Clone repo
!git clone https://github.com/06sahar06/The-Impact-of-Weights-Quantization-in-Fake-Image-Detection.git
%cd /kaggle/working/The-Impact-of-Weights-Quantization-in-Fake-Image-Detection

Cloning into 'The-Impact-of-Weights-Quantization-in-Fake-Image-Detection'...
remote: Enumerating objects: 181, done.
remote: Total 181 (delta 0), reused 0 (delta 0), pack-reused 181 (from 1)
Receiving objects: 100% (181/181), 109.17 MiB | 45.66 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/kaggle/working/The-Impact-of-Weights-Quantization-in-Fake-Image-Detection


In [6]:
%%writefile txt2img.py
import argparse, os, glob, random, gc, torch
from PIL import Image

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('TORCH_COMPILE_DISABLE', '1')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from diffusers import (
    AutoPipelineForImage2Image,
    AutoPipelineForText2Image,
    StableDiffusion3Pipeline,
    FluxPipeline,
    FluxTransformer2DModel,
    AutoencoderKL,
)
from transformers import BitsAndBytesConfig, T5EncoderModel

MODELS = {
    'sdxl': 'stabilityai/stable-diffusion-xl-base-1.0',
    'sd3':  'stabilityai/stable-diffusion-3-medium-diffusers',
    'sd35': 'stabilityai/stable-diffusion-3.5-medium',
    'flux': 'black-forest-labs/FLUX.1-schnell',
    'sd15': 'runwayml/stable-diffusion-v1-5',
}

QUANTIZATION_LEVELS = {'fp16': 'fp16', 'fp8': 'fp8', 'fp4': 'fp4'}
OFFLOAD_DIR = os.environ.get('FLUX_OFFLOAD_DIR', '/tmp/offload_flux')
os.makedirs(OFFLOAD_DIR, exist_ok=True)


def get_bnb_config(level):
    if level == 'fp16': return None
    if level == 'fp8':  return BitsAndBytesConfig(load_in_8bit=True)
    if level == 'fp4':  return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
    )
    raise ValueError(f'Unsupported: {level}')


def flush():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def load_flux_transformer(model_id, bnb_config, dtype):
    return FluxTransformer2DModel.from_pretrained(
        model_id,
        subfolder='transformer',
        quantization_config=bnb_config,
        torch_dtype=dtype,
        device_map='auto',
        low_cpu_mem_usage=True,
    )


class ImageGenerator:
    def __init__(self, model_key, quantization, device=None):
        self.model_key    = model_key
        self.quantization = quantization
        self.pipeline     = None
        self.pipeline_t2i = None
        self.device = (device or
                       ('cuda' if torch.cuda.is_available() else
                        'mps'  if torch.backends.mps.is_available() else 'cpu'))
        print(f'Device: {self.device}')
        self.load_pipeline()

    def load_pipeline(self):
        print(f'Loading {self.model_key} [{self.quantization}]...')
        flush()
        model_id   = MODELS[self.model_key]
        bnb_config = get_bnb_config(self.quantization)
        dtype      = torch.float16

        # ── FLUX ─────────────────────────────────────────────────────────────
        if self.model_key == 'flux':
            # T5 in 8bit on CPU — 9.4GB in fp16, must stay on CPU
            t5_bnb = BitsAndBytesConfig(load_in_8bit=True)
            print('Loading T5 text encoder in 8bit on CPU...')
            text_encoder_2 = T5EncoderModel.from_pretrained(
                model_id,
                subfolder='text_encoder_2',
                quantization_config=t5_bnb,
                torch_dtype=dtype,
                device_map='cpu',
            )
            flush()
            print('T5 loaded. Loading transformer...')

            # transformer in 8bit on CUDA via device_map='auto'
            # fp16 mode: 8bit transformer (fp16 true doesn't fit in VRAM)
            # fp8 mode:  8bit transformer
            t8 = BitsAndBytesConfig(load_in_8bit=True)
            transformer = load_flux_transformer(
                model_id,
                t8 if self.quantization == 'fp16' else bnb_config,
                dtype,
            )
            flush()
            print('Transformer loaded. Assembling pipeline...')

            # No .to(), no enable_model_cpu_offload — both conflict with
            # bitsandbytes quantized modules
            self.pipeline_t2i = FluxPipeline.from_pretrained(
                model_id,
                transformer=transformer,
                text_encoder_2=text_encoder_2,
                torch_dtype=dtype,
                low_cpu_mem_usage=True,
            )
            flush()

            # Move VAE to CUDA
            self.pipeline_t2i.vae = self.pipeline_t2i.vae.to('cuda')
            print('VAE moved to CUDA')

            # Wrap VAE decode to move latents to CUDA before decoding
            # Latents from transformer stay on CPU in this hybrid setup
            # enable_model_cpu_offload cannot fix this because it calls .to()
            # on quantized modules which is not supported
            _old_decode = self.pipeline_t2i.vae.decode
            def _decode_cuda(latents, *args, **kwargs):
                return _old_decode(latents.to('cuda'), *args, **kwargs)
            self.pipeline_t2i.vae.decode = _decode_cuda
            print('VAE decode patched to move latents to CUDA')

            self.pipeline_t2i.enable_attention_slicing(1)
            try:
                self.pipeline_t2i.vae.enable_slicing()
                self.pipeline_t2i.vae.enable_tiling()
            except Exception:
                pass

        # ── SDXL ─────────────────────────────────────────────────────────────
        elif self.model_key == 'sdxl':
            if self.quantization == 'fp16':
                self.pipeline_t2i = AutoPipelineForText2Image.from_pretrained(
                    model_id, torch_dtype=dtype,
                    variant='fp16', use_safetensors=True,
                )
            else:
                vae = AutoencoderKL.from_pretrained(
                    model_id, subfolder='vae', torch_dtype=torch.float32,
                )
                self.pipeline_t2i = AutoPipelineForText2Image.from_pretrained(
                    model_id, vae=vae,
                    quantization_config=bnb_config,
                    torch_dtype=dtype, use_safetensors=True,
                )
            self.pipeline_t2i.enable_model_cpu_offload()

        # ── SD3 / SD3.5 ───────────────────────────────────────────────────────
        elif self.model_key in ('sd3', 'sd35'):
            if self.quantization == 'fp16':
                self.pipeline_t2i = StableDiffusion3Pipeline.from_pretrained(
                    model_id, torch_dtype=dtype, low_cpu_mem_usage=True,
                )
            else:
                self.pipeline_t2i = StableDiffusion3Pipeline.from_pretrained(
                    model_id, quantization_config=bnb_config,
                    torch_dtype=dtype, low_cpu_mem_usage=True,
                )
            self.pipeline_t2i.enable_model_cpu_offload()
            self.pipeline_t2i.enable_attention_slicing()

        # ── SD1.5 ─────────────────────────────────────────────────────────────
        else:
            if self.quantization == 'fp16':
                self.pipeline_t2i = AutoPipelineForText2Image.from_pretrained(
                    model_id, torch_dtype=dtype, use_safetensors=True,
                ).to(self.device)
            else:
                self.pipeline_t2i = AutoPipelineForText2Image.from_pretrained(
                    model_id, quantization_config=bnb_config,
                    torch_dtype=dtype, use_safetensors=True,
                )
                self.pipeline_t2i.enable_model_cpu_offload()
            self.pipeline_t2i.enable_attention_slicing()

        self.pipeline = AutoPipelineForImage2Image.from_pipe(self.pipeline_t2i)
        print(f'{self.model_key} [{self.quantization}] ready.')

    def generate(self, prompt, image=None, strength=0.3,
                 guidance_scale=0.0, steps=4, seed=None):
        generator = torch.Generator('cpu').manual_seed(
            seed if seed is not None else random.randint(0, 2**32-1)
        )
        kwargs = dict(
            prompt=prompt,
            num_inference_steps=steps,
            guidance_scale=guidance_scale,
            generator=generator,
        )
        if image is not None:
            kwargs['image']    = image
            kwargs['strength'] = strength
            print('Starting generation...', flush=True)
            result = self.pipeline(**kwargs).images[0]
            print('Pipeline finished, saving...', flush=True)
            return result
        print('Starting generation...', flush=True)
        result = self.pipeline_t2i(**kwargs).images[0]
        print('Pipeline finished, saving...', flush=True)
        return result


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--mode',         type=str,   choices=['img2img','txt2img'], required=True)
    parser.add_argument('--prompt',       type=str,   default='')
    parser.add_argument('--prompts',      nargs='+',  default=None)
    parser.add_argument('--models',       nargs='+',  default=['sdxl'], choices=MODELS.keys())
    parser.add_argument('--quantization', nargs='+',  default=['fp16'], choices=QUANTIZATION_LEVELS.keys())
    parser.add_argument('--input_dir',    type=str,   default='input_images')
    parser.add_argument('--output_dir',   type=str,   default='output_txt2img')
    parser.add_argument('--strength',     type=float, default=0.3)
    parser.add_argument('--steps',        type=int,   default=4)
    parser.add_argument('--guidance',     type=float, default=0.0)
    parser.add_argument('--device',       type=str,   default=None)
    parser.add_argument('--seed',         type=int,   default=123)
    parser.add_argument('--seeds',        nargs='+',  type=int, default=None)
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)
    prompts = args.prompts if args.prompts else [args.prompt]
    seeds   = args.seeds   if args.seeds   else [args.seed]

    input_images = []
    if args.mode == 'img2img':
        input_images = sorted(
            glob.glob(os.path.join(args.input_dir, '*.jpg')) +
            glob.glob(os.path.join(args.input_dir, '*.png'))
        )
        if not input_images:
            print(f'No images in {args.input_dir}'); return

    total = len(args.models) * len(args.quantization) * len(prompts) * len(seeds)
    done  = 0

    for model_key in args.models:
        for quant in args.quantization:
            try:
                gen = ImageGenerator(model_key, quant, device=args.device)
            except Exception as e:
                import traceback
                traceback.print_exc()
                flush(); continue

            if args.mode == 'txt2img':
                for p_idx, prompt in enumerate(prompts):
                    for seed in seeds:
                        out = f'{args.output_dir}/{model_key}_{quant}_p{p_idx}_seed{seed}.png'
                        if os.path.exists(out):
                            print(f'Skipping: {out}'); done += 1; continue
                        try:
                            img = gen.generate(
                                prompt, steps=args.steps,
                                guidance_scale=args.guidance, seed=seed,
                            )
                            img.save(out)
                            del img; flush()
                            done += 1
                            print(f'[{done}/{total}] Saved: {out}')
                        except Exception as e:
                            import traceback
                            traceback.print_exc()
                            flush()
            else:
                for img_path in input_images:
                    stem = os.path.splitext(os.path.basename(img_path))[0]
                    for p_idx, prompt in enumerate(prompts):
                        for seed in seeds:
                            out = f'{args.output_dir}/{stem}_{model_key}_{quant}_p{p_idx}_seed{seed}.png'
                            if os.path.exists(out):
                                print(f'Skipping: {out}'); done += 1; continue
                            try:
                                src = Image.open(img_path).convert('RGB')
                                img = gen.generate(
                                    prompt, image=src, strength=args.strength,
                                    steps=args.steps, guidance_scale=args.guidance, seed=seed,
                                )
                                img.save(out)
                                del img, src; flush()
                                done += 1
                                print(f'[{done}/{total}] Saved: {out}')
                            except Exception as e:
                                import traceback
                                traceback.print_exc()
                                flush()
            del gen; flush()


if __name__ == '__main__':
    main()

Overwriting txt2img.py


In [13]:
#CELL: Set resume point — edit these two values based on your next generation

RESUME_PROMPT = 3   # which prompt to start from (0, 1, 2, or 3)
RESUME_SEED   = 133 # which seed to start from

import os

OUTPUT = '/kaggle/working/flux_fp16'
os.makedirs(OUTPUT, exist_ok=True)

prompts_count = 4
seeds = list(range(123, 173))

created = 0
for p in range(prompts_count):
    for s in seeds:
        # Stop creating placeholders once we reach the resume point
        if p == RESUME_PROMPT and s == RESUME_SEED:
            break
        fname = f'{OUTPUT}/flux_fp16_p{p}_seed{s}.png'
        if not os.path.exists(fname):
            open(fname, 'w').close()
            created += 1
    if p == RESUME_PROMPT:
        break

real = [f for f in os.listdir(OUTPUT) if os.path.getsize(os.path.join(OUTPUT, f)) > 0]
skip = [f for f in os.listdir(OUTPUT) if os.path.getsize(os.path.join(OUTPUT, f)) == 0]
print(f'Placeholders created: {created}')
print(f'Will skip: {len(skip)} images')
print(f'Will generate: {len(seeds) * prompts_count - len(os.listdir(OUTPUT))} images')
print(f'Starting from: flux_fp16_p{RESUME_PROMPT}_seed{RESUME_SEED}.png')

Placeholders created: 36
Will skip: 155 images
Will generate: 40 images
Starting from: flux_fp16_p3_seed133.png


In [14]:
# CELL 6: Generate FLUX fp16 — 200 images
import os
os.environ['HF_TOKEN']                  = 'hf_token'
os.environ['HUGGING_FACE_HUB_TOKEN']    = 'hf_token'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF']   = 'expandable_segments:True'
os.environ['FLUX_OFFLOAD_DIR']          = '/kaggle/working/offload_flux'

OUTPUT = '/kaggle/working/flux_fp16'
os.makedirs(OUTPUT, exist_ok=True)

!python txt2img.py \
  --mode txt2img \
  --models flux \
  --quantization fp16 \
  --prompts \
    "a high-resolution portrait photograph, realistic lighting, DSLR, shallow depth of field" \
    "an indoor scene with warm lighting, wooden furniture, cozy atmosphere, photorealistic" \
    "a serene landscape with mountains and a lake, golden hour, photorealistic" \
    "a close-up of a human face, studio lighting, ultra sharp, 8k resolution" \
  --seeds 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 \
          143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 \
          163 164 165 166 167 168 169 170 171 172 \
  --steps 4 \
  --guidance 0.0 \
  --output_dir "$OUTPUT"

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Device: cuda
Loading flux [fp16]...
Loading T5 text encoder in 8bit on CPU...
Loading checkpoint shards: 100%|██████████████████| 2/2 [01:13<00:00, 36.64s/it]
T5 loaded. Loading transformer...
Loading checkpoint shards: 100%|██████████████████| 3/3 [01:34<00:00, 31.60s/it]
Transformer loaded. Assembling pipeline...
Loading pipeline components...: 100%|█████████████| 7/7 [00:02<00:00,  2.71it/s]
VAE moved to CUDA
VAE decode patched to move latents to CUDA
flux [fp16] ready.
Skipping: /kaggle/working/flux_fp16/flux_fp16_p0_seed123.png
Skipping: /kaggle/working/flux_fp16/flux_fp16_p0_seed124.png
Skipping: /kaggle/working/flux_fp16/flux_fp16_p0_seed125.png
Skipping: /kaggle/working/flux_fp16/flux_

In [8]:
# CELL 7: Generate FLUX fp8 — 200 images
import os
os.environ['HF_TOKEN']                  = 'hf_token'
os.environ['HUGGING_FACE_HUB_TOKEN']    = 'hf_token'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF']   = 'expandable_segments:True'
os.environ['FLUX_OFFLOAD_DIR']          = '/kaggle/working/offload_flux'

OUTPUT = '/kaggle/working/flux_fp8'
os.makedirs(OUTPUT, exist_ok=True)

!python txt2img.py \
  --mode txt2img \
  --models flux \
  --quantization fp8 \
  --prompts \
    "a high-resolution portrait photograph, realistic lighting, DSLR, shallow depth of field" \
    "an indoor scene with warm lighting, wooden furniture, cozy atmosphere, photorealistic" \
    "a serene landscape with mountains and a lake, golden hour, photorealistic" \
    "a close-up of a human face, studio lighting, ultra sharp, 8k resolution" \
  --seeds 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 \
          143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 \
          163 164 165 166 167 168 169 170 171 172 \
  --steps 4 \
  --guidance 0.0 \
  --output_dir "$OUTPUT"

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Device: cuda
Loading flux [fp8]...
Loading T5 text encoder in 8bit on CPU...
config.json: 100%|█████████████████████████████| 782/782 [00:00<00:00, 5.03MB/s]
model.safetensors.index.json: 19.9kB [00:00, 57.6MB/s]
text_encoder_2/model-00001-of-00002.safe(…):   0%|  | 0.00/4.99G [00:00<?, ?B/s]
text_encoder_2/model-00001-of-00002.safe(…):   0%|  | 0.00/4.99G [00:00<?, ?B/s]
text_encoder_2/model-00001-of-00002.safe(…):   0%|  | 0.00/4.99G [00:00<?, ?B/s]
text_encoder_2/model-00001-of-00002.safe(…):   0%| | 92.9k/4.99G [00:00<2:58:37,
text_encoder_2/model-00001-of-00002.safe(…):   0%| | 3.06M/4.99G [00:01<14:47, 5
text_encoder_2/model-00001-of-00002.safe(…):   0%| | 6.66M/4.99G [00:01<08:23, 9
tex

In [ ]:
# CELL 8: Verify and zip for download
import glob, os, zipfile

for quant in ['fp16', 'fp8']:
    folder = f'/kaggle/working/flux_{quant}'
    files  = glob.glob(f'{folder}/*.png')
    print(f'flux_{quant}: {len(files)}/200 images')
    if len(files) < 200:
        existing = set(os.path.basename(f) for f in files)
        missing  = [
            f'flux_{quant}_p{p}_seed{s}.png'
            for p in range(4) for s in range(123, 173)
            if f'flux_{quant}_p{p}_seed{s}.png' not in existing
        ]
        print(f'  Missing {len(missing)}, e.g.: {missing[:3]}')

# Zip both folders for easy download from Kaggle output
for quant in ['fp16', 'fp8']:
    folder = f'/kaggle/working/flux_{quant}'
    zippath = f'/kaggle/working/flux_{quant}.zip'
    with zipfile.ZipFile(zippath, 'w') as z:
        for f in glob.glob(f'{folder}/*.png'):
            z.write(f, os.path.basename(f))
    print(f'Zipped: {zippath} ({os.path.getsize(zippath)/1e6:.0f} MB)')